# 10 — What happens near zero? The five tests, on MDPL2 halos

The default velocity-centered run (notebook 05) bins `[0, 1)` h⁻¹ Mpc as a **single shell**.
Everything the pipeline knows about the inner region is that one number, and it is
volume-weighted toward its outer edge — so the question *"what does ζ̂₁ do as r → 0?"* is not
merely unanswered there, it is unaskable. This notebook resolves that bin into seven and asks
whether what appears is a measurement.

Two questions, and only a resolved ladder can separate them:

1. **Is the innermost bin a measurement at all**, or one draw of a heavy-tailed statistic? A
   stack over N_c centers whose per-center contributions have no effective variance ceiling
   can read anything and still look like a curve.
2. **If it is, what is the one-halo regime doing?** On the `full` catalog that bin is
   populated by host–subhalo pairs whose velocities are *virial, not infall*
   (`velocity_centered._default_shells`' comment block). §C of `DM_simulation.ipynb` found, on
   the Gadget-4 *particle* field, that the most crowded centers contribute with the **opposite
   sign** inside ~0.2 h⁻¹ Mpc — virial cancellation. Halos are a different tracer, sparser and
   biased; whether the same thing happens to them is a separate measurement.

`dvcorr.pipeline.robustness` holds the recipe; §C ran it on particles. **This notebook runs the
identical five tests on the MDPL2 halo field** — the population every science result in this
project is actually measured on. Nothing is implemented here: as everywhere in `notebooks/`,
this only wires library stage functions together.

| # | test | what it asks | function |
|---|---|---|---|
| 1 | **precision ladder** | does the scatter fall as 1/√N_c, and does the quoted SEM match it? | `precision_ladder` |
| 2 | **bootstrap interval** | what does a *distribution-free* interval say, given the terms are not Gaussian? | `bootstrap_confidence` |
| 3 | **trim ladder** | how much of the answer rides on its largest few contributors? | `trim_ladder` |
| 4 | **seed repeats** | does a freshly drawn centre set reproduce it? | `seed_repeats` |
| 5 | **mechanism split** | does the variable the *explanation* names actually order the result? | `crowding_split`, `speed_split` |

Tests 1, 2, 3 and 5 cost **no estimator pass at all**: they are re-sums of `per_center_dipole`,
`per_center_count` and `per_center_speed`, arrays the estimator already returned — the same
recombination discipline every null in this package follows. `restack(stack)` over all centres
reproduces `normalize_stacked_dipole` exactly (pinned in `tests/test_robustness.py`), which is
what licenses plotting a subset curve on the same axes as the published one. Only test 4 runs
the estimator, once per seed.

### Three deliberate choices

**Shells stop at r = 10, not 64.** The question is at small r, and the outer shells cost almost
all of the runtime: a `query_ball_point` scales as r³, so `r_max = 10` holds ~260× fewer
tracers per ball than notebook 05's 64. The core cut is also far kinder — a centre needs
`r_max` of clearance inside the carve, so 90% of candidates survive here against 48% there.
The two together are what make a centre set an order of magnitude larger than the pipeline's
affordable, and more centres is exactly what question 1 needs.

**The inner floor is a pair count, not a length.** MDPL2's force softening (5 h⁻¹ kpc) and its
mass resolution both sit ~100× below the default `min_radius = 1.0`, so neither is what stops
the ladder — the binding constraint is statistical, and 1.0 itself came from requiring ≥ 50
expected uniform pairs on the sparse `mvir12` catalog. This run uses the **same rule** at its
own n̄ and its own N_c, which lands two orders of magnitude further in. Derived below, never
typed in.

**Observer frame, real space.** This is notebook 05's configuration — the one the pipeline
publishes and the one whose inner bin is in question. Notebook 06's velocity frame and notebook
07's redshift space each have their own inner-bin behaviour; neither is what is being audited
here.

In [1]:
%matplotlib inline

import numpy as np

from dvcorr import conventions
from dvcorr.config import PathsConfig, ShellConfig, SPACING_LOG
from dvcorr.pipeline.velocity_centered import (
    RunConfig,
    load_and_carve,
    box_number_density,
    draw_candidates_from_arrays,
    select_shared_centers,
    run_estimator,
)
from dvcorr.pipeline.robustness import (
    RobustnessRunConfig,
    per_center_stack,
    restack,
    merge_shells,
    precision_ladder,
    bootstrap_confidence,
    trim_ladder,
    seed_repeats,
    crowding_shell_index,
    crowding_split,
    speed_split,
    make_robustness_figure,
)

observer = np.asarray(conventions.OBSERVER_POSITION, dtype=float)
paths = PathsConfig()

print("observer (box center):", conventions.OBSERVER_POSITION)
print("BOX_SIZE            :", conventions.BOX_SIZE, "h^-1 Mpc")

observer (box center): [500. 500. 500.]
BOX_SIZE            : 1000.0 h^-1 Mpc


## 1 — The tracer field

`base` is notebook 05's configuration, unmodified, and it is used for exactly two things: the
carve (`sub_volume_radius`, `catalog`) and, later, the coarse `[0, 1)` bin this notebook is
resolving. Loading through it rather than through a bespoke config is what makes the numbers
below comparable to the published run rather than merely similar to it.

The catalog is the **full** MDPL2 Rockstar snapshot — 127M halos down to 2 particles, subhalos
included, no mass cut (`dvcorr.config.catalog`). That choice is the reason the inner bins exist
at all: on `mvir12` (distinct halos only) every centre carries a hard exclusion hole of its
host's virial radius, and r < 1 h⁻¹ Mpc is genuinely empty. Here it is not.

*Runtime: the load is a few minutes and ~16 GB of RAM.*

In [2]:
base = RunConfig()
carved = load_and_carve(base, paths)

# n_bar is the cosmic mean over the WHOLE box, not the density inside the carve
# -- see notebook 05 step 4 for why the sub-volume's own density would leave a
# stray 1 + delta_sub multiplying every shell.
n_bar = box_number_density(carved.n_total)
mean_halo_spacing = n_bar ** (-1.0 / 3.0)

print()
print(f"catalog            : {base.catalog.name}")
print(f"n_bar              : {n_bar:.5f} halos / (h^-1 Mpc)^3")
print(f"mean halo spacing  : {mean_halo_spacing:.3f} h^-1 Mpc   (n_bar^(-1/3))")
print(f"subhalo fraction   : {1.0 - carved.is_distinct.mean():.1%} of the carved population")

loading mdpl2_rockstar_snapnum125.parquet [full, no mass cut, subhalos included] ...


loaded 127388160 halos (0 removed by in-memory cuts)


carved 14418601 of 127388160 halos within sub_volume_radius = 300.0 h^-1 Mpc (11.32%)



catalog            : full
n_bar              : 0.12739 halos / (h^-1 Mpc)^3
mean halo spacing  : 1.987 h^-1 Mpc   (n_bar^(-1/3))
subhalo fraction   : 12.1% of the carved population


## 2 — Parameters, and where the inner floor comes from

The floor is **derived from this run's own n̄ and N_c**, by the rule that set the pipeline
default: the smallest radius at which the whole stack expects at least `MIN_EXPECTED_PAIRS`
tracer pairs *if the field were uniform*,

$$\bar n\,\frac{4\pi}{3}r^3\,N_c \;\ge\; \texttt{MIN\_EXPECTED\_PAIRS}.$$

`velocity_centered`'s comment block applies exactly this at 50 pairs to `mvir12`
(n̄ ≈ 4e-3) and gets r ≥ 0.94 — which is where `min_radius = 1.0` came from. The full catalog
is ~30× denser and this run has ~14× more centres, so the same rule lands two orders of
magnitude further in.

It is a **uniform-field** expectation, deliberately: clustering only ever adds pairs, so the
bound holds whatever ξ does, and the realized occupancy the estimator reports (printed in step
3) is the factor by which clustering beats it. N_c is not known until the core cut has run, so
the geometric survival fraction `((R_sub − r_max)/R_sub)³` stands in for it — an underestimate
of the pairs, hence a conservative floor.

`crowding_shell` points at the bin bracketing the **mean halo spacing** n̄^(−1/3), not at the
shell under test. Splitting centres on the occupancy of the very shell whose dipole is being
measured is circular — a centre with no tracers there contributes exactly zero to it by
construction — and once the inner shells are resolved, most centres fall in that trivial group.
The spacing is the smallest scale at which a halo count is a density rather than a coincidence.

In [3]:
# The pair-count floor. 50 is the threshold `velocity_centered`'s comment block
# already used to justify min_radius = 1.0 on mvir12; reused here rather than
# re-chosen, so the two floors are the same rule at different n_bar and N_c.
MIN_EXPECTED_PAIRS = 50.0
R_MAX = 10.0
N_CANDIDATE_CENTERS = 120_000

# N_c is unknown until the core cut runs, so stand in the GEOMETRIC survival
# fraction. It underestimates neither cut nor pairs in a way that could soften
# the floor: fewer centres -> fewer pairs -> a larger (more conservative) r.
core_radius = base.sub_volume_radius - R_MAX
n_centers_expected = N_CANDIDATE_CENTERS * (core_radius / base.sub_volume_radius) ** 3
r_floor = (
    3.0 * MIN_EXPECTED_PAIRS / (4.0 * np.pi * n_bar * n_centers_expected)
) ** (1.0 / 3.0)

robust_shells = ShellConfig(
    min_radius=r_floor,
    max_radius=R_MAX,
    spacing=SPACING_LOG,
    n_bins=12,
    include_zero_bin=True,
)
edges = robust_shells.shell_edges

# Environment measured at the mean halo spacing, NOT in the shell under test.
crowding_shell = int(np.searchsorted(edges, mean_halo_spacing, side="right") - 1)

rbcfg = RobustnessRunConfig(
    sub_volume_radius=base.sub_volume_radius,
    catalog=base.catalog,
    shells=robust_shells,
    n_candidate_centers=N_CANDIDATE_CENTERS,
    trim_rank_shell=0,           # the bin nearest zero -- what tests 3 and 5 report on
    crowding_shell=crowding_shell,
    figure_max_radius=3.0,       # plot-only zoom; every table still spans all shells
)

print(f"expected N_c       : {n_centers_expected:,.0f}"
      f"  ({100 * (core_radius / base.sub_volume_radius) ** 3:.1f}% of candidates;"
      f" notebook 05, at r_max = {base.shells.max_radius:g}:"
      f" {100 * ((base.sub_volume_radius - base.shells.max_radius) / base.sub_volume_radius) ** 3:.1f}%)")
print(f"r_floor            : {r_floor:.4f} h^-1 Mpc"
      f"   <- {MIN_EXPECTED_PAIRS:.0f} expected uniform pairs over the stack")
print(f"  notebook 05's     : {base.shells.min_radius:g} h^-1 Mpc"
      f"   ({base.shells.min_radius / r_floor:.0f}x further out)")
print()
print("shell_edges        :", np.round(edges, 4))
print(f"shells below {base.shells.min_radius:g}   :",
      int((edges < base.shells.min_radius).sum()) - 1,
      "  (notebook 05 had 1: the single [0, 1) bin)")
print()
print("trim_rank_shell    :", rbcfg.trim_rank_shell,
      f"-> [{edges[0]:g}, {edges[1]:.4f}) h^-1 Mpc")
print("crowding_shell     :", crowding_shell,
      f"-> [{edges[crowding_shell]:.3f}, {edges[crowding_shell + 1]:.3f}) h^-1 Mpc"
      f"   (brackets {mean_halo_spacing:.3f})")
print("figure_max_radius  :", rbcfg.figure_max_radius, "h^-1 Mpc (view only)")
print()
print("bootstrap resamples:", rbcfg.n_bootstrap, " confidence:", rbcfg.confidence)
print("subset fractions   :", rbcfg.subset_fractions, f"x {rbcfg.n_subset_draws} draws each")
print("trim counts        :", rbcfg.trim_counts)
print("seed repeats       :", rbcfg.n_seed_repeats)

expected N_c       : 108,396  (90.3% of candidates; notebook 05, at r_max = 64: 48.7%)
r_floor            : 0.0953 h^-1 Mpc   <- 50 expected uniform pairs over the stack
  notebook 05's     : 1 h^-1 Mpc   (10x further out)

shell_edges        : [ 0.      0.0953  0.1404  0.2069  0.3049  0.4494  0.6623  0.976   1.4384
  2.1198  3.1241  4.6042  6.7854 10.    ]
shells below 1   : 7   (notebook 05 had 1: the single [0, 1) bin)

trim_rank_shell    : 0 -> [0, 0.0953) h^-1 Mpc
crowding_shell     : 8 -> [1.438, 2.120) h^-1 Mpc   (brackets 1.987)
figure_max_radius  : 3.0 h^-1 Mpc (view only)

bootstrap resamples: 2000  confidence: 0.95
subset fractions   : (0.02, 0.05, 0.1, 0.25, 0.5, 1.0) x 64 draws each
trim counts        : (0, 1, 10, 100, 1000)
seed repeats       : 4


## 3 — One estimator pass

Tracers are **all** carved halos, exactly as in notebook 05 — only the centre draw and the
binning differ. Everything downstream then hangs off **one** `PerCenterStack`: the estimator's
per-centre arrays plus the cached normalization, bundled so that every test below re-sums the
same numbers rather than re-deriving them.

`realized / expected` in the printout is the occupancy ratio 1 + ξ_hh(r) — how far clustering
beats the uniform-field expectation the floor was set from. It is the reason a bin holding
50 *expected* pairs is not a bin holding 50 pairs.

In [4]:
rb_candidates = draw_candidates_from_arrays(
    rbcfg, carved.pos, carved.vel, carved.mvir, carved.is_distinct
)
print("candidate centers:", rb_candidates.s.shape[0])
print()

rb_centers = select_shared_centers(rbcfg, rb_candidates.s, rb_candidates.v, observer)
print()

rb_result = run_estimator(
    rbcfg, rb_centers.s_centers, rb_centers.v_centers, carved.pos, observer
)

stack = per_center_stack(rb_result, n_bar)
full = restack(stack)

realized = stack.per_center_count.sum(axis=0)
expected = stack.expected_occupancy * stack.n_centers

print()
print(f"N_c      : {stack.n_centers:,}"
      f"   ({stack.n_centers / n_centers_expected:.2f}x the geometric prediction above)")
print("r_cen    :", np.round(stack.shell_centers, 3))
print("zeta_hat :", np.round(full.zeta_hat, 1))
print("sem      :", np.round(full.sem, 1))
print("sem/|z|  :", np.round(full.sem / np.abs(full.zeta_hat), 3))
print("monopole :", np.round(full.monopole_norm, 1))   # hard rule 6
print()
print("expected :", np.round(expected, 0))
print("realized :", realized)
print("1 + xi   :", np.round(realized / expected, 1))

candidate centers: 120000

n_candidates = 120000 -> core cut (core_margin = 10.0) -> n_core = 108743 (90.6% survive) -> speed floor (min_center_speed = 1.0 km/s) -> n_centers = 108743 (0 dropped as too slow)



n_candidates = 108743, n_centers = 108743 (100.0% survive the core cut)

N_c      : 108,743   (1.00x the geometric prediction above)
r_cen    : [0.048 0.118 0.174 0.256 0.377 0.556 0.819 1.207 1.779 2.622 3.864 5.695
 8.393]
zeta_hat : [201.8  97.9 197.3 234.6 203.8 215.2 185.3 162.2 135.1  98.2  70.4  50.6
  37.7]
sem      : [322.1 190.1  94.4  49.8  27.5  16.8  11.6   8.1   5.1   2.6   1.3   0.7
   0.4]
sem/|z|  : [1.596 1.941 0.478 0.212 0.135 0.078 0.062 0.05  0.038 0.027 0.019 0.013
 0.011]
monopole : [31216.4 19666.2 14724.3 11169.2  8309.2  6056.3  4180.1  2674.6  1597.4
   947.8   627.6   469.7   381.7]

expected : [5.0000000e+01 1.1000000e+02 3.5300000e+02 1.1310000e+03 3.6210000e+03
 1.1589000e+04 3.7095000e+04 1.1873800e+05 3.8006700e+05 1.2165560e+06
 3.8940700e+06 1.2464514e+07 3.9897617e+07]
realized : [5.1980000e+03 7.0390000e+03 1.5740000e+04 3.6298000e+04 8.2595000e+04
 1.8285900e+05 3.9179600e+05 7.9970000e+05 1.5875260e+06 3.2748250e+06
 7.5067090e+06 1.8988406e+07 5

## 4 — Test 1: the precision ladder

Draw random centre subsets of increasing size, `n_subset_draws` of each, and watch two numbers
per rung: the **across-draw spread** (how much a stack that size actually moves) and the mean
**internal SEM** (how much it *says* it moves). Subsets are drawn without replacement, because
they stand in for smaller runs and a smaller run would not measure the same centre twice.

Three things to read off it:

- **`zeta_hat_mean` should be flat in N_c.** A subset mean is a mean, so it is unbiased at
  every rung by construction. A *trend* here would not be noise — it would mean the subsets are
  not exchangeable with the parent, i.e. a selection leaked into the draw.
- **spread ≈ SEM says the quoted error bar is honest** — but read that comparison at the
  *small* rungs. Sampling without replacement carries the finite-population correction
  √((N_full − N)/(N_full − 1)), so the spread necessarily sits ~√(1 − N/N_full) below the
  reference at large fractions: a subset that is half the parent shares half its centres with it
  and cannot scatter freely. That deficit is arithmetic, not good news. Test 4 is the version
  with no such correction.
- **The 1/√N_c reference line** turns "is this bin real?" into "how many centres would settle
  it?", which is a question with an answer you can act on. The last lines price notebook 05's
  own centre count on it.

The full rung's across-draw spread is exactly zero, on purpose: without replacement there is
only one way to draw the whole set. It stays in the table and is left off the log-scale panel,
where a zero has no place.

In [5]:
ladder = precision_ladder(rbcfg, stack)
b = rbcfg.trim_rank_shell

print(f"Innermost shell [{stack.shell_edges[0]:g}, {stack.shell_edges[1]:.4f}) h^-1 Mpc\n")
print(f"{'N_c':>8}  {'<zeta_hat>':>12}  {'across-draw':>12}  {'internal SEM':>13}  {'1/sqrt(N) ref':>14}")
for i, n in enumerate(ladder.n_centers):
    print(f"{n:8d}  {ladder.zeta_hat_mean[i, b]:12.1f}  {ladder.zeta_hat_spread[i, b]:12.1f}"
          f"  {ladder.sem_mean[i, b]:13.1f}  {ladder.reference_spread[i, b]:14.1f}")

# What notebook 05's centre count buys in this bin, priced on this ladder.
# Its N_c is DERIVED from its own config -- the core cut is geometric, so the
# survival fraction is ((R_sub - r_max)/R_sub)**3 and no number is typed in.
n_c_notebook_05 = base.n_candidate_centers * (
    (base.sub_volume_radius - base.shells.max_radius) / base.sub_volume_radius
) ** 3
expected_wander = full.sem[b] * np.sqrt(stack.n_centers / n_c_notebook_05)
print(f"\nnotebook 05: N_c ~ {n_c_notebook_05:,.0f} -> expected 1-sigma wander of "
      f"~{expected_wander:.0f} km/s in this bin")
print(f"this run   : N_c = {stack.n_centers:,} -> {full.sem[b]:.0f} km/s")

Innermost shell [0, 0.0953) h^-1 Mpc

     N_c    <zeta_hat>   across-draw   internal SEM   1/sqrt(N) ref
    2175         420.4        2372.6         2246.1          2277.2
    5437         -25.1        1213.6         1431.3          1440.3
   10874         159.5        1033.1         1003.7          1018.4
   27186         201.0         510.7          644.8           644.1
   54372         237.3         288.1          453.9           455.5
  108743         201.8           0.0          322.1           322.1

notebook 05: N_c ~ 7,789 -> expected 1-sigma wander of ~1203 km/s in this bin
this run   : N_c = 108,743 -> 322 km/s


## 5 — Test 2: the bootstrap interval

The SEM is a Gaussian statement about a sum whose terms are emphatically not Gaussian. The
bootstrap makes no such assumption: resample the **centres** with replacement and read a
percentile interval straight off the resampled stacks.

Centres are the right resampling unit. A centre is what the stack averages over, and its whole
shell — every tracer in it — travels with it, so the resample preserves exactly the within-shell
correlation that makes the per-centre contributions heavy-tailed in the first place. Resampling
tracers instead would break that and quietly report a much smaller interval.

The interval is reported as explicit low/high edges, never as a symmetric ±, because on the
inner shells it is not symmetric. `bootstrap_std` beside `sem` is the compact version of the
same comparison: a large gap between them *is* the non-Gaussianity, which test 3 then measures
directly.

**Comparing back to notebook 05 needs `merge_shells`.** Its `[0, 1)` number is one wide bin;
this run's is seven narrow ones, and they cannot be compared by averaging — each shell's
normalization divides by its *own* n̄V_b, so a plain mean of the fine bins equals nothing in
particular. `merge_shells` does it correctly, summing the raw per-centre numerators and
normalizing once against the merged shell's own volume. That is exactly the number the coarse
run would have produced from these centres, and `tests/test_robustness.py` checks it against a
real coarse-binned estimator pass rather than against a re-derivation.

In [6]:
interval = bootstrap_confidence(rbcfg, stack)

print(f"{interval.confidence:.0%} percentile interval from {interval.n_resamples} resamples\n")
print(f"{'r_lo':>7} {'r_hi':>7}  {'zeta_hat':>10}  {'low':>10}  {'high':>10}  {'boot std':>9}  {'SEM':>9}  {'ratio':>6}")
for i in range(stack.shell_centers.size):
    print(f"{stack.shell_edges[i]:7.3f} {stack.shell_edges[i + 1]:7.3f}"
          f"  {interval.zeta_hat[i]:10.1f}  {interval.low[i]:10.1f}  {interval.high[i]:10.1f}"
          f"  {interval.bootstrap_std[i]:9.1f}  {interval.sem[i]:9.1f}"
          f"  {interval.bootstrap_std[i] / interval.sem[i]:6.2f}")

# --- the like-for-like comparison against notebook 05's coarse bin -----------
n_inner = int((stack.shell_edges < base.shells.min_radius).sum()) - 1
coarse = merge_shells(stack, [0, n_inner, stack.shell_centers.size])
coarse_curve = restack(coarse)
print(f"\nmerging the inner {n_inner} shells into one "
      f"[{coarse.shell_edges[0]:g}, {coarse.shell_edges[1]:.3f}) bin"
      f"   (notebook 05 measured [0, {base.shells.min_radius:g}))")
print(f"  merged : {coarse_curve.zeta_hat[0]:8.1f} +/- {coarse_curve.sem[0]:.1f} km/s"
      f"   (N_c = {coarse.n_centers:,})")

# Does the bin nearest zero sit on the inward continuation of the ones outside
# it? The outer bins rise inward by a near-constant factor, so extrapolating one
# bin further is a fair expectation to test the measurement against.
step = interval.zeta_hat[1] / interval.zeta_hat[2]
print(f"\nbin-to-bin rise just outside the innermost: {step:.2f}x")
print(f"  extrapolated expectation for [{stack.shell_edges[0]:g}, {stack.shell_edges[1]:.4f}): "
      f"{interval.zeta_hat[1] * step:.1f} km/s")
print(f"  measured                                    : {interval.zeta_hat[b]:.1f} km/s"
      f"  [{interval.low[b]:.1f}, {interval.high[b]:.1f}]")

95% percentile interval from 2000 resamples

   r_lo    r_hi    zeta_hat         low        high   boot std        SEM   ratio
  0.000   0.095       201.8      -428.5       815.5      316.3      322.1    0.98
  0.095   0.140        97.9      -274.6       465.9      192.2      190.1    1.01
  0.140   0.207       197.3        10.9       383.5       95.9       94.4    1.02
  0.207   0.305       234.6       131.4       332.7       51.4       49.8    1.03
  0.305   0.449       203.8       150.1       255.1       27.2       27.5    0.99
  0.449   0.662       215.2       184.2       249.0       16.5       16.8    0.98
  0.662   0.976       185.3       162.5       208.2       11.6       11.6    1.01
  0.976   1.438       162.2       146.7       178.1        8.0        8.1    0.99
  1.438   2.120       135.1       125.2       145.0        5.0        5.1    0.98
  2.120   3.124        98.2        93.3       103.4        2.6        2.6    0.98
  3.124   4.604        70.4        68.0        73.1  

## 6 — Test 3: the trim ladder

Drop the k largest-|contribution| centres and restack. Ranking is on the **absolute** value,
both signs: trimming only the positive tail would be a biased estimator wearing a robustness
check's clothes — it would drag every stack downward by construction and make any feature look
fragile.

The ranking comes from **one** shell (`trim_rank_shell`, here the one nearest zero). It has to:
the centre that dominates the inner bin is not the one that dominates the outer bins, and
re-ranking per shell would trim a different sample in every column — which is not a subset of
anything and has no interpretation.

What to expect, and what would be damning:

- The value **will** move. The distribution is heavy-tailed; `mean / median` says how badly —
  except on a shell narrow enough that most centres hold no tracer at all, where the median is
  exactly 0 and `n_contributing` is the number that means something.
- What must survive is the **sign and the order of magnitude**. A stack that halves when you
  drop 0.1% of its centres is still telling you something; a stack that crosses zero is not.
- The **monopole** is printed beside it (hard rule 6) and does double duty here: a trim that
  also moves the monopole has changed which *shells* are represented, not just which velocities,
  and its dipole is then not comparable to the untrimmed one.

In [7]:
trim = trim_ladder(rbcfg, stack)

contributing = trim.n_contributing / trim.n_centers[0]
print(f"ranked on shell [{stack.shell_edges[b]:g}, {stack.shell_edges[b + 1]:.4f}) h^-1 Mpc")
print(f"centers contributing at all: {trim.n_contributing:,} of {trim.n_centers[0]:,}"
      f"  ({contributing:.1%})")
print(f"per-center contribution there: mean = {trim.mean_contribution:.2f} km/s, "
      f"median = {trim.median_contribution:.2f} km/s")
if trim.median_contribution != 0.0:
    print(f"  -> mean/median = {trim.mean_contribution / trim.median_contribution:.0f}x")
else:
    # Not a degenerate case to guard around: most centers are EMPTY in a shell
    # this narrow, so the median IS zero and the informative comparison is
    # against the contributing centers only.
    among = np.median(
        np.abs(stack.per_center_dipole[:, b] * stack.norm_scale[b])[
            stack.per_center_dipole[:, b] != 0.0
        ]
    )
    print(f"  -> median is 0 because {1 - contributing:.1%} of centers are EMPTY here;"
          f" median |contribution| among contributors = {among:.1f} km/s")
print()

print(f"{'dropped':>9}  {'N_c kept':>9}  {'zeta_hat[0]':>12}  {'vs untrimmed':>13}  {'monopole[0]':>12}")
for i, k in enumerate(trim.trim_counts):
    frac = trim.zeta_hat[i, b] / trim.zeta_hat[0, b]
    print(f"{k:9d}  {trim.n_centers[i]:9d}  {trim.zeta_hat[i, b]:12.1f}  {frac:12.1%}"
          f"  {trim.monopole_norm[i, b]:12.1f}")

print("\nfull trimmed curves:")
print("r_cen :", np.round(stack.shell_centers, 3))
for i, k in enumerate(trim.trim_counts):
    print(f"drop {k:>4} :", np.round(trim.zeta_hat[i], 1))

ranked on shell [0, 0.0953) h^-1 Mpc
centers contributing at all: 4,790 of 108,743  (4.4%)
per-center contribution there: mean = 201.78 km/s, median = 0.00 km/s
  -> median is 0 because 95.6% of centers are EMPTY here; median |contribution| among contributors = 206762.9 km/s

  dropped   N_c kept   zeta_hat[0]   vs untrimmed   monopole[0]
        0     108743         201.8        100.0%       31216.4
        1     108742         169.3         83.9%       31180.8
       10     108733          93.5         46.3%       30836.8
      100     108643         145.6         72.2%       28714.8
     1000     107743         115.6         57.3%       18211.5

full trimmed curves:
r_cen : [0.048 0.118 0.174 0.256 0.377 0.556 0.819 1.207 1.779 2.622 3.864 5.695
 8.393]
drop    0 : [201.8  97.9 197.3 234.6 203.8 215.2 185.3 162.2 135.1  98.2  70.4  50.6
  37.7]
drop    1 : [169.3  97.9 197.3 234.6 203.9 215.2 185.2 162.2 135.1  98.2  70.4  50.6
  37.7]
drop   10 : [ 93.5 128.9 200.3 237.7 203.6 214.

## 7 — Test 4: independent centre draws

Every test so far conditions on the centres already in hand. This one redraws them.

`seed_repeats` spawns `n_seed_repeats` child seeds from `RunConfig.seed` — spawned, not
`seed + i`, for the reason `null_realization_seeds` gives: adjacent integer seeds are not
guaranteed to give independent streams, and independence is the entire point of the test. Each
seed re-draws candidates, re-applies the core cut and the speed floor, and re-runs the
estimator. The **tracer field is held fixed**, so what the across-seed spread measures is
centre-draw variance and nothing else.

This is the only expensive test here — one estimator pass per seed — and the only error
statement in the notebook that includes the centre *selection* rather than just the centres
selected. Expect it to exceed test 1's prediction: centres drawn this densely from one carve
share tracers, so they are not independent samples, which is precisely the caveat
`center_standard_error`'s own docstring raises about the SEM.

In [8]:
repeats = seed_repeats(
    rbcfg, carved.pos, carved.vel, carved.mvir, carved.is_distinct,
    carved.pos, observer, n_bar,
)

print()
print("r_cen :", np.round(stack.shell_centers, 3))
for i, s in enumerate(repeats.seeds):
    print(f"N_c = {repeats.n_centers[i]:>7,} :", np.round(repeats.zeta_hat[i], 1))
print("mean  :", np.round(repeats.mean, 1))
print("spread:", np.round(repeats.spread, 1))
print()
print(f"inner bin: {repeats.mean[b]:.1f} +/- {repeats.spread[b]:.1f} km/s across "
      f"{repeats.seeds.size} independent draws")
print(f"  test 1 predicted a 1-sigma wander of ~{full.sem[b]:.1f} km/s at this N_c")
print(f"  ratio (honest / quoted): {repeats.spread[b] / full.sem[b]:.2f}x")
print("  all shells:", np.round(repeats.spread / full.sem, 2))

n_candidates = 120000 -> core cut (core_margin = 10.0) -> n_core = 108559 (90.5% survive) -> speed floor (min_center_speed = 1.0 km/s) -> n_centers = 108559 (0 dropped as too slow)


n_candidates = 108559, n_centers = 108559 (100.0% survive the core cut)


n_candidates = 120000 -> core cut (core_margin = 10.0) -> n_core = 108574 (90.5% survive) -> speed floor (min_center_speed = 1.0 km/s) -> n_centers = 108574 (0 dropped as too slow)


n_candidates = 108574, n_centers = 108574 (100.0% survive the core cut)


n_candidates = 120000 -> core cut (core_margin = 10.0) -> n_core = 108579 (90.5% survive) -> speed floor (min_center_speed = 1.0 km/s) -> n_centers = 108578 (1 dropped as too slow)


n_candidates = 108578, n_centers = 108578 (100.0% survive the core cut)


n_candidates = 120000 -> core cut (core_margin = 10.0) -> n_core = 108566 (90.5% survive) -> speed floor (min_center_speed = 1.0 km/s) -> n_centers = 108566 (0 dropped as too slow)


n_candidates = 108566, n_centers = 108566 (100.0% survive the core cut)

r_cen : [0.048 0.118 0.174 0.256 0.377 0.556 0.819 1.207 1.779 2.622 3.864 5.695
 8.393]
N_c = 108,559 : [256.3 197.6 317.7 138.2 164.  177.6 164.6 163.1 133.2 100.1  69.3  50.
  37.5]
N_c = 108,574 : [746.3 652.6 307.7 182.1 191.9 182.8 164.6 150.4 130.2  96.1  68.   49.1
  37.2]
N_c = 108,578 : [284.  117.  309.  247.6 181.7 183.2 182.3 156.7 128.   96.3  69.8  50.3
  37.4]
N_c = 108,566 : [-160.8  274.3  103.5  218.   190.2  195.4  173.6  152.8  128.6   93.8
   67.    48.6   36.5]
mean  : [281.4 310.4 259.5 196.5 182.  184.8 171.3 155.8 130.   96.6  68.5  49.5
  37.1]
spread: [370.7 237.  104.1  47.2  12.8   7.6   8.5   5.5   2.3   2.6   1.3   0.8
   0.4]

inner bin: 281.4 +/- 370.7 km/s across 4 independent draws
  test 1 predicted a 1-sigma wander of ~322.1 km/s at this N_c
  ratio (honest / quoted): 1.15x
  all shells: [1.15 1.25 1.1  0.95 0.47 0.45 0.74 0.69 0.45 0.99 0.98 1.14 1.08]


## 8 — Test 5: the mechanism split

The four tests above measure *how noisy* the number is. This one tests the *explanation*.

Any physical story about a feature names a per-centre variable — "virialized environments", "the
fastest centres". That makes it falsifiable: the groups must **order** in the direction the
story claims. A fluctuation predicts no ordering at all, so a monotone trend is evidence and a
flat split is not.

Two splits, both free:

- **`crowding_split`** — on tracer count at the mean halo spacing, i.e. the local overdensity of
  the centre's own neighbourhood. If "virial motion cancels the dipole" carries over from
  particles to halos, crowded centres must trend toward **zero and past it**.
- **`speed_split`** — on the weight |u_α|, by percentile. The weight multiplies the whole
  contribution, so the top group's *share of the stack* is, to first order, exactly what a global
  velocity cut at that percentile would remove.

**The crowding is counted in a different shell from the one under test**, for the circularity
reason spelled out in step 2.

Both columns matter. `mean_contribution` is per centre and therefore comparable between groups
of wildly different size; `share_of_stack` says how much of the answer each group actually
supplies. A group can have the largest mean and contribute nothing, if it holds twenty centres.

**The headline shell is the one tests 1–3 just declared unmeasured, so it is not read alone.**
`PropertySplit.mean_contribution` is `(G, B)` — every group, every shell — so the same split is
also printed at the innermost shell whose *bootstrap interval excludes zero* (derived from test
2, not chosen), and then across the whole ladder. That is what turns "the crowded group is
negative" from one number in a noisy bin into a statement with a radial extent: virial
cancellation must switch off somewhere, and the matrix says where.

In [9]:
crowding = crowding_split(rbcfg, stack)
c_shell = crowding_shell_index(rbcfg)

print(f"Centres GROUPED by halo count in [{stack.shell_edges[c_shell]:.3f}, "
      f"{stack.shell_edges[c_shell + 1]:.3f}) h^-1 Mpc  (the environment proxy)")
print(f"Contribution REPORTED in    [{stack.shell_edges[b]:g}, {stack.shell_edges[b + 1]:.4f})"
      " h^-1 Mpc  (the shell under test)\n")
print(f"{'neighbours':>14}  {'N_c':>8}  {'mean contrib':>13}  {'share':>8}  {'monopole':>10}")
for i, lab in enumerate(crowding.group_labels):
    print(f"{lab:>14}  {crowding.n_centers[i]:8d}  {crowding.mean_contribution[i, b]:13.1f}"
          f"  {crowding.share_of_stack[i, b]:8.1%}  {crowding.monopole_norm[i, b]:10.1f}")

speed = speed_split(rbcfg, stack)
print(f"\nSplit on the weight |u| ({rbcfg.speed_percentiles} percentile edges)\n")
print(f"{'|u| band (km/s)':>26}  {'N_c':>8}  {'mean contrib':>13}  {'share':>8}")
for i, lab in enumerate(speed.group_labels):
    print(f"{lab:>26}  {speed.n_centers[i]:8d}  {speed.mean_contribution[i, b]:13.1f}"
          f"  {speed.share_of_stack[i, b]:8.1%}")

# What a global velocity cut would cost, measured rather than assumed: restack
# without the top percentile group and compare, shell by shell.
kept = np.flatnonzero(
    stack.per_center_speed < np.percentile(stack.per_center_speed, rbcfg.speed_percentiles[-2])
)
cut_curve = restack(stack, kept)
print(f"\ndropping the top {100 - rbcfg.speed_percentiles[-2]:g}% of |u| "
      f"({stack.n_centers - kept.size} centers):")
print(f"  bin nearest zero {full.zeta_hat[b]:.1f} -> {cut_curve.zeta_hat[b]:.1f} km/s "
      f"({cut_curve.zeta_hat[b] / full.zeta_hat[b] - 1:+.1%})")
print("  all shells:", np.round(cut_curve.zeta_hat / full.zeta_hat - 1.0, 3))

# --- the same split, read where the measurement holds ------------------------
# The innermost shell whose bootstrap interval excludes zero -- DERIVED from
# test 2 rather than picked, so the choice cannot be tuned to the answer.
resolved = (interval.low > 0.0) | (interval.high < 0.0)
b_resolved = int(np.argmax(resolved))
print(f"\n\nSame split, read at the innermost shell test 2 resolves: "
      f"[{stack.shell_edges[b_resolved]:.4f}, {stack.shell_edges[b_resolved + 1]:.4f}) "
      f"h^-1 Mpc")
print(f"  zeta_hat = {interval.zeta_hat[b_resolved]:.1f} km/s"
      f"  [{interval.low[b_resolved]:.1f}, {interval.high[b_resolved]:.1f}]\n")
print(f"{'neighbours':>14}  {'N_c':>8}  {'mean contrib':>13}  {'share':>8}")
for i, lab in enumerate(crowding.group_labels):
    print(f"{lab:>14}  {crowding.n_centers[i]:8d}"
          f"  {crowding.mean_contribution[i, b_resolved]:13.1f}"
          f"  {crowding.share_of_stack[i, b_resolved]:8.1%}")

print("\nmean contribution per centre, every group x every shell (km/s):")
print(f"{'neighbours':>14} " + "".join(f"{r:>8.3g}" for r in stack.shell_centers))
for i, lab in enumerate(crowding.group_labels):
    print(f"{lab:>14} " + "".join(
        f"{v:>8.0f}" for v in crowding.mean_contribution[i]
    ))

Centres GROUPED by halo count in [1.438, 2.120) h^-1 Mpc  (the environment proxy)
Contribution REPORTED in    [0, 0.0953) h^-1 Mpc  (the shell under test)

    neighbours       N_c   mean contrib     share    monopole
        [0, 1)      4364         -266.8     -5.3%      5927.5
        [1, 3)     13950         -262.5    -16.7%     13199.1
       [3, 10)     43630          229.8     45.7%     23964.9
      [10, 30)     34535          532.3     83.8%     38721.8
     [30, 100)     10745          647.4     31.7%     55701.5
    [100, 300)      1510        -6135.2    -42.2%    133115.2
    [300, inf)         9        73930.1      3.0%    245717.4

Split on the weight |u| ((0.0, 50.0, 90.0, 99.0, 99.9, 100.0) percentile edges)

           |u| band (km/s)       N_c   mean contrib     share
    [0.000592361, 211.833)     54371           31.7      7.9%
         [211.833, 557.13)     43497          139.8     27.7%
         [557.13, 1034.21)      9787          172.1      7.7%
        [1034.21, 

## 9 — All five, plus the monopole companion as its own figure

`make_robustness_figure` returns a `DipoleFigures(dipole, monopole)`
(CLAUDE.md hard rule 6): `.dipole` is the 5-panel sheet below, reading across
then down: **(1)** the precision ladder in the shell under test, the one
panel here with logarithmic axes — its x-axis is N_c, not r, and a power law
is what it exists to show; **(2)** the curve with its bootstrap band and, for
contrast, its Gaussian SEM band; **(3)** the trim ladder; **(4)** the
independent draws with their across-seed band; **(5)** the crowding split —
bars grouped by environment at the mean halo spacing, their heights the
contribution to the bin nearest zero, annotated with each group's share.
`.monopole` is a separate, smaller figure: the monopole companion (hard rule
6) — never merged into the same image as the dipole panels.

The four radial panels are **zoomed to r ≤ 3** (`figure_max_radius`). Every
curve on them still carries all thirteen shells — the limit trims the view,
never the measurement, and the tables above report the full range. Every
radial curve is now a **step**, held flat across each bin's full width from
r = 0 (CLAUDE.md hard rule 10), not a point at the shell midpoint.

Radial axes stay cartesian — log *binning* never implies log *axes* in this
project, and `tests/test_robustness.py` pins that here the same way
`tests/test_plot_wiring.py` pins it for the pipeline figures.

In [ ]:
# DipoleFigures(dipole, monopole) -- two separate figures, never one merged
# image (CLAUDE.md hard rule 6); %matplotlib inline renders both.
robust_figs = make_robustness_figure(
    rbcfg, stack, ladder, interval, trim, repeats, crowding
)

## 10 — The verdict

Three questions came in. Two get clean answers, one gets a boundary — and the boundary is the
most useful thing here, because it is exactly where the answer would have been most interesting.

### 1. Notebook 05's `[0, 1)` bin is a measurement; its inner half is not

Merged back to one bin for a like-for-like comparison (`merge_shells`, step 5), the inner seven
shells read **193.9 ± 11.7 km/s** over `[0, 0.976)` at N_c = 108,743. That is a 17σ number and
the coarse pipeline is entitled to it. Resolved, though, it is not one thing:

| r (h⁻¹ Mpc) | 0.048 | 0.118 | 0.174 | 0.256 | 0.377 | 0.556 | 0.819 |
|---|---|---|---|---|---|---|---|
| ζ̂₁ | 202 | 98 | 197 | 235 | 204 | 215 | 185 |
| SEM | 322 | 190 | 94 | 50 | 28 | 17 | 12 |

Everything from r ≈ 0.14 outward is measured — the bootstrap interval first excludes zero in
`[0.140, 0.207)`, at 197.3 [10.9, 383.5]. The two bins inside that are not: `[0, 0.095)` reads
202 with a 95% interval of **[−429, +816]**, **95.6% of centres hold no tracer there at all**,
and dropping the **ten** largest contributors out of 108,743 halves it (202 → 94). Test 1 prices
the cure — the scatter is clean 1/√N_c over a 50× range, so ±100 km/s in that bin wants ~10×
more centres.

Worth noting what did *not* limit it. The floor was set at 50 expected **uniform** pairs
(r = 0.0953 h⁻¹ Mpc); clustering delivered **5,198**, a factor 1 + ξ = 104. The innermost bin is
not short of pairs, it is short of *centres holding them* — the pairs pile onto a few thousand
crowded centres, and a mean over 108,743 centres of which 4.4% contribute is a mean with a very
heavy tail.

### 2. The rise stops at r ≈ 0.6, and a single `[0, 1)` bin cannot show that

Outward from the plateau the curve is unambiguous and tightly measured: 215 → 185 → 162 → 135 →
98 → 70 → 51 → 38 km/s from r = 0.56 to 8.4, every bin at ≥ 13σ. Inward of ~0.6 it **stops
rising**: 215 ± 17 at r = 0.56, 204 ± 28 at 0.38, 235 ± 50 at 0.26 are one value within their
errors. So ζ̂₁ flattens at **~200–215 km/s between r ≈ 0.14 and 0.66** rather than continuing to
climb, and below 0.14 nothing can be said at this N_c.

The coarse bin is not *wrong* about the amplitude — 193.9 lands squarely inside the plateau —
but it cannot distinguish a plateau from a continued rise, and those are different statements
about the one-halo term.

### 3. The environment ordering is monotone at every measured shell — no cancellation

Grouping centres by their halo count at the mean halo spacing (1.44–2.12 h⁻¹ Mpc, measured
*away* from the shell under test) and reading the per-centre contribution at r = 0.56 h⁻¹ Mpc,
the innermost shell where the split and the curve are both solid:

| neighbours | [0,1) | [1,3) | [3,10) | [10,30) | [30,100) | [100,300) |
|---|---|---|---|---|---|---|
| N_c | 4364 | 13950 | 43630 | 34535 | 10745 | 1510 |
| mean contribution | −12 | 34 | 86 | 234 | 619 | 2831 |

Strictly monotone, over three orders of magnitude in contribution — and the same ordering holds
at every shell from r = 0.26 out to r = 3.9. The `[300, ∞)` group is **nine centres** and is not
a group; ignore its row. The monopole climbs monotonically with crowding too (5,927 → 245,717),
which is the check that the proxy is ordering by environment and not by something else.

**What is not here is §C's turnover.** On the Gadget-4 particle field, the most crowded centres
contributed with the *opposite sign* inside 0.2 h⁻¹ Mpc — virial motion cancelling the dipole.
The `[100, 300)` group does go negative here, at −6,135 and −3,428 km/s — but **only in the two
bins section 1 just declared unmeasured**, and it is positive (+4,221 … +2,312) at every shell
outside them. Two adjacent noisy bins agreeing in sign is not evidence; the `[30, 100)` group
posts a lone −325 at r = 0.17 and is positive on both sides of it, which is what that kind of
scatter looks like.

So the honest statement is not "halos do not show virial cancellation". It is: **if halos show
it, it lives inside r ≈ 0.14 h⁻¹ Mpc, and this catalog at this N_c cannot see in there.** That
is a scale statement, and it is testable — the same run at 10× the centres would settle it.

### 4. Here the SEM is honest — unlike on the particle field

`bootstrap_std / sem` is **0.98–1.03 on every one of the thirteen shells**, and the across-seed
spread (test 4, the only error statement that includes the centre *selection*) is 1.15× the
quoted SEM in the innermost bin and 0.45–1.25× elsewhere. §C found 1.3–1.6× on Gadget-4
particles and warned that every error bar in that notebook needed the factor applied.

The difference is centre density, not tracer density. §C packed 96,113 centres into a core of
radius 120 h⁻¹ Mpc (1.3e-2 per h⁻³ Mpc³); this run spreads 108,743 over radius 290
(1.1e-3, **12× sparser**), so its centres' shells overlap far less and are much closer to the
independent samples `center_standard_error` assumes. **Error bars on this notebook can be read
as quoted.**

### 5. What a velocity cut would cost

The 99th–99.9th percentile band of |u| — 979 centres, 0.9% of the set — supplies **56.5%** of
the innermost bin, but the top 0.1% supplies only 0.2% and dropping it moves that bin by −0.1%.
At the shells that are actually measured, dropping the top 0.1% costs a steady **8–10%** between
r ≈ 0.5 and 2 h⁻¹ Mpc. So a global speed cut is not free, and its price is concentrated in a
band rather than in the extreme tail: cutting at the 99.9th percentile is cheap, cutting at the
99th would remove more than half the inner-bin signal.

### What this leaves open

- **Below r ≈ 0.14 h⁻¹ Mpc is unmeasured, not empty.** 10× the centres would resolve it, and
  the cost is known: the estimator pass is linear in N_c and this one took well under a minute.
  Nothing about the box, the resolution, or the softening (5 h⁻¹ kpc, ~19× below the floor used
  here) is in the way — only N_c.
- **The plateau's shape is not pinned.** 235 / 204 / 215 at r = 0.26 / 0.38 / 0.56 is a plateau
  within errors, but so is a shallow peak, and so is a shallow dip. Distinguishing them is the
  same 10×.
- **Subhalos are 12.1% of the carved population and are what populates r < 1 at all.** Whether
  the plateau is a property of the matter field or of Rockstar's subhalo assignment inside hosts
  is a question this notebook cannot separate — `08_halo_class_comparison.ipynb` is the arm that
  can.